In [1]:
import torch
import torch.nn as nn
from transformers import (
    CLIPVisionModel,
    CLIPVisionConfig,
    ViTModel,
    T5EncoderModel,
    T5ForConditionalGeneration,
)

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image

import pandas as pd
import torch
from torch.utils.data import Dataset
# from Retrieval_module import Retriever

class BlipRAGDataset(Dataset):
    def __init__(
        self,
        image_dir, 
        metadata_path,
        retriever,
        caption_prompt="Describe the image.",
        num_similar_captions=3,
        transform=None,
    ):
        """
        metadata_path: path to JSON metadata file
        similar_indices: dict[int, list[int]] from FAISS
        image_base_path: base directory for images
        caption_prompt: base captioning instruction
        num_similar_captions: how many retrieved captions to inject
        transform: optional image transform
        """
        self.retriever = retriever
        
        with open(metadata_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

        # self.metadata_df = pd.DataFrame(metadata)
        
        self.image_base_path = Path(image_dir)
        self.caption_prompt = caption_prompt
        self.num_similar_captions = num_similar_captions
        self.transform = transform

        # assert "image_name" in self.metadata_df.columns
        # assert "captions" in self.metadata_df.columns
        # assert "similiar_images" in self.metadata_df.columns
                    

    def __len__(self):
        # return len(self.metadata_df)
        return len(self.metadata)

    def _load_image(self, image_name):
        path = self.image_base_path / image_name
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

    def _sample_similar_captions(self, neighbors):
        if len(neighbors) == 0:
            return []

        k = min(self.num_similar_captions, len(neighbors))
        sampled = neighbors[:k]
        
        captions = []
        for idx in sampled:
            caps = self.retriever.retrieve_captions(idx)
            captions.append(random.choice(caps))

        return captions

    def _build_prompt(self, similar_captions):
        if len(similar_captions) == 0:
            return self.caption_prompt

        context = "\n".join(f"- {c}" for c in similar_captions)

        return (
            f"Similar images are described as:\n"
            f"{context} \n\n "
            f"{self.caption_prompt}"
        )

    def __getitem__(self, idx):
        img_data = self.metadata[str(idx)]

        image = self._load_image(img_data["image_name"])
        target_caption = max(img_data["captions"], key=len)
        similar_captions = self._sample_similar_captions(img_data['similar_images'])
        prompt = self._build_prompt(similar_captions)

        return {
            "image": image,
            "prompt": prompt,
            "target_caption": target_caption,
        }



class BlipDataCollator:
    def __init__(self, processor, max_seq_len=128, device="cuda"):
        self.processor = processor
        self.device = device
        self.max_seq_len = max_seq_len

    def __call__(self, batch):
        images = [b["image"] for b in batch]
        prompts = [b["prompt"] for b in batch]
        targets = [b["target_caption"] for b in batch]

        inputs = self.processor(
            images=images,
            text=prompts,
            padding="max_length",
            padding_side='left',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        )

        labels = self.processor.tokenizer(
            targets,
            padding="max_length",
            padding_side='left',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        ).input_ids

        # Important: ignore padding tokens in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        inputs["labels"] = labels

        # return {k: v.to(self.device) for k, v in inputs.items()}
        return {k: v for k, v in inputs.items()}

In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
import torch
from transformers import T5TokenizerFast



class VLMDataset(Dataset):
    def __init__(
        self,
        image_dir,
        ref_image_dir,
        metadata_path,
        retriever,
        caption_prompt="A Picture of",
        num_similar_captions=3,
        transform=None,
    ):
        """
        image_dir: folder with all images
        metadata_path: JSON with {idx: {"image_name": str, "captions": List[str], "similar_images": List[int]}}
        retriever: object with method retrieve_captions(idx) -> List[str]
        """
        self.retriever = retriever
        self.image_base_path = Path(image_dir)
        self.ref_image_base_path = Path(ref_image_dir)
        self.caption_prompt = caption_prompt
        self.num_similar_captions = num_similar_captions
        self.transform = transform

        with open(metadata_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

    def __len__(self):
        return len(self.metadata)

    def _load_image(self, image_name, base_path):
        path = base_path / image_name
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

    def _sample_similar_captions(self, neighbors):
        if len(neighbors) == 0:
            return []

        k = min(self.num_similar_captions, len(neighbors))
        sampled = random.sample(neighbors, k)
        captions = []
        for idx in sampled:
            caps = self.retriever.retrieve_captions(idx)
            captions.append(random.choice(caps))
        return captions

    def _build_prompt(self, similar_captions):
        if len(similar_captions) == 0:
            return self.caption_prompt
        context = "\n".join(f"- {c}" for c in similar_captions)
        return f"Similar images are described as:\n{context}\n\n{self.caption_prompt}"

    def __getitem__(self, idx):
        img_data = self.metadata[str(idx)]
        
        # Query image
        query_image = self._load_image(img_data["image_name"], self.image_base_path)
        
        # Retrieved image (take first similar image)
        retrieved_idx = img_data["similar_images"][0]
        retrieved_image_name = self.retriever.retreive_image_name(retrieved_idx)
        retrieved_image = self._load_image(retrieved_image_name, self.ref_image_base_path)
        
        # Retrieved captions
        retrieved_captions = self._sample_similar_captions(img_data["similar_images"])
        
        # Target caption (longest one)
        target_caption = max(img_data["captions"], key=len)
        
        # Prompt for training
        prompt = self._build_prompt(retrieved_captions)
        
        return {
            "query_image": query_image,
            "retrieved_image": retrieved_image,
            "retrieved_captions": retrieved_captions,
            "prompt": prompt,
            "target_caption": target_caption,
        }


class VLMDataCollator:
    def __init__(self, processor, tokenizer, max_seq_len=128, device="cuda"):
        """
        processor: a HuggingFace processor with vision & text capabilities
        """
        self.processor = processor
        self.tokenizer = tokenizer
        self.device = device
        self.max_seq_len = max_seq_len

    def __call__(self, batch):
        query_images = [b["query_image"] for b in batch]
        retrieved_images = [b["retrieved_image"] for b in batch]
        prompts = [b["prompt"] for b in batch]
        targets = [b["target_caption"] for b in batch]

        # Process images (stack query + retrieved along batch dimension)
        # Some VLMs expect separate keys for query and retrieved images
        pixel_values = self.processor(
            images=query_images,
            return_tensors="pt"
        ).pixel_values

        retrieved_pixel_values = self.processor(
            images=retrieved_images,
            return_tensors="pt"
        ).pixel_values

        # Process text prompts
        inputs = self.tokenizer(
            prompts,
            padding="longest",
            padding_side='right',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        )

        labels = self.tokenizer(
            targets,
            padding="longest",
            padding_side='right',
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        ).input_ids
        

        # ignore padding tokens in loss
        labels[labels == self.tokenizer.tokenizer.pad_token_id] = -100

        return {
            "query_pixel_values": pixel_values.to(self.device),
            "retrieved_pixel_values": retrieved_pixel_values.to(self.device),
            "input_ids": inputs.input_ids.to(self.device),
            "attention_mask": inputs.attention_mask.to(self.device),
            "labels": labels.to(self.device),
        }
        
        
# batch = next(iter(dataloader))
# batch.keys()

# dict_keys([
#   'pixel_values',            # query images
#   'retrieved_pixel_values',  # retrieved images
#   'input_ids',               # prompt tokens
#   'attention_mask',          # prompt masks
#   'labels'                   # target captions
# ])



In [ ]:
tokenizer = T5TokenizerFast.from_pretrained("t5-base")


{'input_ids': [71, 212, 18283, 10488, 388, 5119, 3, 9, 710, 18, 7, 109, 15, 162, 26, 1001, 8677, 11, 3, 9, 2164, 18, 7, 157, 14029, 26, 2335, 5119, 3, 9, 3, 7, 109, 15, 162, 924, 3270, 33, 464, 44, 3, 9, 26982, 3, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [40]:

# class FusionBlock(nn.Module):
#     def __init__(self, hidden_dim, num_heads):
#         super().__init__()
#         self.cross_attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim, num_heads=num_heads, batch_first=True
#         )
#         self.ln1 = nn.LayerNorm(hidden_dim)
#         self.ff = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim * 4),
#             nn.GELU(),
#             nn.Linear(hidden_dim * 4, hidden_dim),
#         )
#         self.ln2 = nn.LayerNorm(hidden_dim)

#     def forward(self, text_hidden, vision_hidden):
#         attn_out, _ = self.cross_attn(
#             query=text_hidden, key=vision_hidden, value=vision_hidden
#         )
#         x = self.ln1(text_hidden + attn_out)
#         x = self.ln2(x + self.ff(x))
#         return x
    
    
class FusionBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.self_attn = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )

        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ln3 = nn.LayerNorm(hidden_dim)

        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )

    def forward(self, text_hidden, vision_hidden):
        # cross-attention
        x, _ = self.cross_attn(
            query=text_hidden, key=vision_hidden, value=vision_hidden
        )
        x = self.ln1(text_hidden + x)

        # self-attention
        sa, _ = self.self_attn(x, x, x)
        x = self.ln2(x + sa)

        # feed-forward
        x = self.ln3(x + self.ff(x))
        return x



class MultimodalVLM(nn.Module):
    def __init__(
        self,
        vision_encoder_name: str,
        text_encoder_name: str,
        text_decoder_name: str,
        num_fusion_blocks: int,
        num_heads: int = 8,
        use_local_files=True
    ):
        super().__init__()

        # Vision encoder
        vision_config = CLIPVisionConfig()
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_encoder_name, config=vision_config, local_files_only=use_local_files)
        self.vision_dim = self.vision_encoder.config.hidden_size

        # Text encoder
        self.text_encoder = T5EncoderModel.from_pretrained(text_encoder_name, local_files_only=use_local_files)
        self.text_dim = self.text_encoder.config.d_model

        # Text decoder (LM head)
        self.text_decoder = T5ForConditionalGeneration.from_pretrained(text_decoder_name, local_files_only=use_local_files)

        # Project vision → text space
        self.vision_proj = nn.Linear(self.vision_dim, self.text_dim)

        # Fusion blocks
        self.fusion_blocks = nn.ModuleList(
            [FusionBlock(self.text_dim, num_heads) for _ in range(num_fusion_blocks)]
        )

    def forward(
        self,
        query_pixel_values,
        retrieved_pixel_values,
        input_ids,
        attention_mask,
        labels=None,
    ):
        # Encode images
        # q_vis = self.vision_encoder(query_pixel_values).last_hidden_state

        # r_vis = self.vision_encoder(retrieved_pixel_values).last_hidden_state

        # vision_hidden = torch.cat([q_vis, r_vis], dim=1)
        # vision_hidden = self.vision_proj(vision_hidden)
        
        q_vis = self.vision_encoder(query_pixel_values).last_hidden_state

        if retrieved_pixel_values is not None:
            r_vis = self.vision_encoder(retrieved_pixel_values).last_hidden_state
            vision_hidden = torch.cat([q_vis, r_vis], dim=1)
        else:
            vision_hidden = q_vis

        vision_hidden = self.vision_proj(vision_hidden)


        # Encode text
        text_hidden = self.text_encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

        # Fusion
        for block in self.fusion_blocks:
            text_hidden = block(text_hidden, vision_hidden)

        # Decode
        outputs = self.text_decoder(
            inputs_embeds=text_hidden,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )

        return outputs

In [77]:
# openai/clip-vit-base-patch16
# google/vit-base-patch16-224

# Instantiate and count parameters
model = MultimodalVLM(
    # vision_encoder_name="openai/clip-vit-base-patch16",
    vision_encoder_name="openai/clip-vit-base-patch32",
    text_encoder_name="t5-base",
    text_decoder_name="t5-base",
    num_fusion_blocks=4,
    use_local_files=True
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

Total parameters: 458,385,792


In [78]:
def freeze_module(module: torch.nn.Module):
    for p in module.parameters():
        p.requires_grad = False

# Freeze vision encoder
freeze_module(model.vision_encoder)

# Freeze text encoder
freeze_module(model.text_encoder)

# Freeze text decoder
# freeze_module(model.text_decoder)

In [79]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        # Decoder self-attention (language modeling)
        "SelfAttention.q",
        "SelfAttention.v",

        # Decoder cross-attention (fusion output → text)
        "EncDecAttention.q",
        "EncDecAttention.v",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model.text_decoder = get_peft_model(model.text_decoder, lora_config)
model.text_decoder.print_trainable_parameters()


trainable params: 3,538,944 || all params: 226,442,496 || trainable%: 1.5628


In [80]:
# Unfreeze the lm head for token generation
for param in model.text_decoder.lm_head.parameters():
    print(param.shape[0]*param.shape[1])
    param.requires_grad = True

24674304


In [81]:
import torch
import torch.nn as nn

def print_model_param_stats(model: nn.Module):
    total_params = 0
    trainable_params = 0
    frozen_params = 0

    print(f"{'Module':40} {'Total':>12} {'Trainable':>12} {'Frozen':>12}")
    print("-" * 80)

    # Iterate over top-level modules
    for name, module in model.named_children():
        module_total = sum(p.numel() for p in module.parameters())
        module_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        module_frozen = module_total - module_trainable

        total_params += module_total
        trainable_params += module_trainable
        frozen_params += module_frozen

        print(f"{name:40} {module_total:12,} {module_trainable:12,} {module_frozen:12,}")

    print("-" * 80)
    print(f"{'TOTAL':40} {total_params:12,} {trainable_params:12,} {frozen_params:12,}")


print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                              109,628,544            0  109,628,544
text_decoder                              226,442,496   28,213,248  198,229,248
vision_proj                                   590,592      590,592            0
fusion_blocks                              37,807,104   37,807,104            0
--------------------------------------------------------------------------------
TOTAL                                     461,924,736   66,610,944  395,313,792


In [ ]:
import torch
from torch.amp import autocast, GradScaler

# Optimizer for all trainable params
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# GradScaler for FP16 stability
scaler = GradScaler()

for batch in dataloader:
    images = batch["image"].to(device)
    text_inputs = batch["text"].to(device)
    
    optimizer.zero_grad()
    
    # Forward pass in mixed precision
    with autocast():   # everything inside uses FP16 automatically
        # Example forward
        text_hidden = text_encoder(text_inputs)
        vision_hidden = vision_encoder(images)
        for block in fusion_blocks:
            text_hidden = block(text_hidden, vision_hidden)
        outputs = t5_decoder(input_ids=None, encoder_outputs=text_hidden)
        loss = outputs.loss
    
    # Backward pass
    scaler.scale(loss).backward()
    
    # Step optimizer
    scaler.step(optimizer)
    scaler.update()


In [26]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import LoraConfig, get_peft_model

# --------------------------
# 1. Model & tokenizer
# --------------------------
t5_model_name = "t5-small"  # can be t5-base or t5-large
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device)

# --------------------------
# 2. LoRA config
# --------------------------
lora_config = LoraConfig(
    r=8,                        # rank of LoRA matrices
    lora_alpha=32,               # scaling factor
    target_modules=["q", "v"],   # apply LoRA to attention Q and V
    lora_dropout=0.1,            # dropout in LoRA layers
    bias="none",                 # keep original bias
    task_type="SEQ_2_SEQ_LM"     # seq2seq language modeling
)

# --------------------------
# 3. Wrap T5 with LoRA
# --------------------------
model = get_peft_model(model, lora_config)
print("LoRA model ready. Trainable parameters:")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"{trainable_params:,} / {total_params:,} trainable")

# --------------------------
# 4. Example forward pass
# --------------------------
input_text = "translate English to French: Hello, how are you?"
inputs = tokenizer(input_text, return_tensors="pt").to(device)
labels = tokenizer("Bonjour, comment ça va ?", return_tensors="pt").input_ids.to(device)

outputs = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, labels=labels)
print("Loss:", outputs.loss.item())


ImportError: 
T5Tokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"Trainable params: {trainable:,}")
print(f"Frozen params: {frozen:,}")